Import & Setup

In [2]:
import pandas as pd
import numpy as np
import json, gzip, os, sys, re
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import sqlite3
import gc
import pyarrow.parquet as pq

In [3]:
ROOT_DIR      = Path().resolve().parent
REVIEW_PATH   = ROOT_DIR / "data" / "raw" / "Clothing_Shoes_and_Jewelry.jsonl.gz"
META_PATH     = ROOT_DIR / "data" / "raw" / "meta_Clothing_Shoes_and_Jewelry.jsonl.gz"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
SAMPLE_DIR    = ROOT_DIR / "data" / "sample"
FIGURES_DIR   = ROOT_DIR / "outputs" / "figures"
FINAL_META_CSV = PROCESSED_DIR / "meta_clean_all.csv"
FINAL_REVIEW_CSV = PROCESSED_DIR / "review_clean_all.csv"
MERGED_OUTPUT_CSV = PROCESSED_DIR / "amazon_clothing_full_merged.csv"
CHUNK_SIZE = 500000

Tiền xử lý review

In [5]:
def preprocess_review(df):
    df = df.copy()
    log = {}  # Ghi lại số dòng sau mỗi bước
    log['1_raw'] = len(df)

    # ----------------------------------------------------------
    # BƯỚC A: Chọn cột cần thiết
    # Chỉ giữ các cột có ý nghĩa cho phân tích
    # ----------------------------------------------------------
    needed = ['rating', 'title', 'user_id','text',
              'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
    existing_cols = [c for c in needed if c in df.columns]
    df = df[existing_cols]
    print(f"A. Chọn cột: giữ lại {existing_cols}")

    # ----------------------------------------------------------
    # BƯỚC B: Xử lý missing values
    # rating và title là BẮT BUỘC → drop nếu thiếu
    # các cột khác → fill giá trị mặc định
    # ----------------------------------------------------------
    df = df.dropna(subset=['rating', 'title'])
    df['helpful_vote']      = df.get('helpful_vote', pd.Series(0)).fillna(0)
    df['verified_purchase'] = df.get('verified_purchase', pd.Series(False)).fillna(False)
    log['2_drop_na'] = len(df)
    print(f"B. Drop NA  : {log['1_raw']:,} → {log['2_drop_na']:,} (-{log['1_raw']-log['2_drop_na']:,})")

    # ----------------------------------------------------------
    # BƯỚC C: Chuẩn hóa kiểu dữ liệu
    # ----------------------------------------------------------
    df['rating']        = pd.to_numeric(df['rating'], errors='coerce')
    df['helpful_vote']  = pd.to_numeric(df['helpful_vote'], errors='coerce').fillna(0).astype(int)
    df = df.dropna(subset=['rating'])  # drop nếu rating không parse được
    df['rating']        = df['rating'].astype(float)
    log['3_dtype'] = len(df)
    print(f"C. Dtype    : {log['2_drop_na']:,} → {log['3_dtype']:,}")

    # ----------------------------------------------------------
    # BƯỚC D: Chuyển timestamp → datetime
    # timestamp gốc là Unix milliseconds (số ms từ 1970)
    # ----------------------------------------------------------
    if 'timestamp' in df.columns:
        df['date']  = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
        df['year']  = df['date'].dt.year
        df['month'] = df['date'].dt.month
        print(f"D. Timestamp: range {df['year'].min():.0f} - {df['year'].max():.0f}")

    # ----------------------------------------------------------
    # BƯỚC E: Tạo nhãn Sentiment từ Rating
    # 4-5 sao → positive | 3 sao → neutral | 1-2 sao → negative
    # Đây là SUPERVISED LABEL cho bài toán Sentiment Analysis
    # ----------------------------------------------------------
    def to_sentiment(r):
        if r >= 4:   return 'positive'
        elif r == 3: return 'neutral'
        else:        return 'negative'

    df['sentiment']       = df['rating'].apply(to_sentiment)
    df['sentiment_score'] = df['rating'].apply(lambda r: 1 if r >= 4 else (0 if r == 3 else -1))
    print(f"E. Sentiment: {df['sentiment'].value_counts().to_dict()}")

    # ----------------------------------------------------------
    # BƯỚC F: Lọc review quá ngắn (noise)
    # Review < 10 ký tự không có giá trị phân tích NLP
    # ----------------------------------------------------------
    if 'text' in df.columns:
        df['text_length'] = df['text'].astype(str).str.len()
        df = df[df['text_length'] >= 10] # Giả sử MIN_TEXT_LENGTH = 10
        
        # QUAN TRỌNG: Xóa cột 'text' và 'text_length' sau khi lọc xong
        df = df.drop(columns=['text', 'text_length'])

    # ----------------------------------------------------------
    # BƯỚC G: Loại bỏ duplicate
    # Cùng 1 user review cùng 1 sản phẩm → chỉ giữ lần đầu
    # ----------------------------------------------------------
    before = len(df)
    df = df.drop_duplicates(subset=['user_id', 'parent_asin'], keep='first')
    log['5_dedup'] = len(df)
    print(f"G. Duplicate: {before:,} → {log['5_dedup']:,} (-{before-log['5_dedup']:,} duplicates)")

    # ----------------------------------------------------------
    # BƯỚC H: Reset index
    # ----------------------------------------------------------
    df = df.reset_index(drop=True)

    print(f"\n✅ Kết quả cuối: {len(df):,} reviews")
    return df, log



Tiền xử lý meta

In [6]:
def preprocess_meta(df):
    df = df.copy()
    print(f"Raw shape: {df.shape}")

    # ----------------------------------------------------------
    # BƯỚC A: Chọn cột cần thiết
    # ----------------------------------------------------------
    needed = ['parent_asin', 'title', 'description',
              'categories', 'average_rating', 'rating_number',
              'store', 'main_category','images','price']
    existing_cols = [c for c in needed if c in df.columns]
    df = df[existing_cols]
    print(f"A. Giữ cột: {existing_cols}")

    # ----------------------------------------------------------
    # BƯỚC B: Drop sản phẩm không có ID hoặc tên
    # ----------------------------------------------------------
    before = len(df)
    df = df.dropna(subset=['parent_asin', 'title'])
    df = df.drop_duplicates(subset=['parent_asin'], keep='first')
    print(f"B. Drop NA/dup: {before:,} → {len(df):,}")

    # ----------------------------------------------------------
    # BƯỚC C: Xử lý Price
    # Giá có thể là '$29.99' hoặc '29.99' → cần chuẩn hóa
    # Lọc outlier: giá <= 0 hoặc > $10,000 là bất thường
    # ----------------------------------------------------------
    if 'price' in df.columns:
        df['price'] = df['price'].astype(str).str.replace(r'[^\d.]', '', regex=True)
        df['price'] = pd.to_numeric(df['price'], errors='coerce')
        
        # Lọc: Chỉ giữ sản phẩm có giá hợp lý (ví dụ > 0)
        df = df[df['price'] > 0] 
        
        # QUAN TRỌNG: Xóa cột 'price' sau khi lọc
        df = df.drop(columns=['price'])

    # ----------------------------------------------------------
    # BƯỚC D: Xử lý Description (list → string)
    # description gốc là list các đoạn văn
    # ----------------------------------------------------------
    if 'description' in df.columns:
        df['description'] = df['description'].apply(
            lambda x: ' '.join(x) if isinstance(x, list) else str(x) if pd.notna(x) else ''
        )
        print(f"D. Description: đã chuyển list → string")

    # ----------------------------------------------------------
    # BƯỚC E: Xử lý Categories
    # categories là list lồng nhau → lấy level 1 làm main_category
    # ----------------------------------------------------------
    if 'categories' in df.columns:
        def extract_category(cats):
            if isinstance(cats, list) and len(cats) > 0:
                inner = cats[0] if isinstance(cats[0], list) else cats
                return inner[0] if len(inner) > 0 else 'Unknown'
            return 'Unknown'
        df['main_category'] = df['categories'].apply(extract_category)
        print(f"E. Categories top 5: {df['main_category'].value_counts().head().to_dict()}")

    df = df.reset_index(drop=True)
    print(f"\n✅ Meta kết quả: {df.shape}")
    return df

Đọc+ tiền xử lý và lưu file review

In [7]:
def process_review_with_sentiment_chunks(filepath, chunk_size=500000):
    print(f"📦 Đang xử lý Review (Sentiment) theo cụm {chunk_size:,}...")
    file_exists = False
    
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        current_chunk = []
        for i, line in enumerate(tqdm(f, desc="Streaming Review")):
            try:
                current_chunk.append(json.loads(line.strip()))
            except: continue

            if len(current_chunk) == chunk_size:
                # Chuyển thành DF và Tiền xử lý (Gọi hàm ở Cell 2.4 của bạn)
                df_temp, _ = preprocess_review(pd.DataFrame(current_chunk))
                
                # Lưu luôn vào file CSV (Append)
                df_temp.to_csv(FINAL_REVIEW_CSV, mode='a', index=False, header=not file_exists)
                
                file_exists = True
                current_chunk = [] 
                del df_temp

        # Lưu phần còn lại
        if current_chunk:
            df_temp, _ = preprocess_review(pd.DataFrame(current_chunk))
            df_temp.to_csv(FINAL_REVIEW_CSV, mode='a', index=False, header=not file_exists)

    print(f"✅ Đã lưu xong file Review cho Sentiment tại: {FINAL_REVIEW_CSV}")

# Chạy bước 1
process_review_with_sentiment_chunks(REVIEW_PATH, CHUNK_SIZE)

📦 Đang xử lý Review (Sentiment) theo cụm 500,000...


Streaming Review: 485176it [00:04, 118159.88it/s]

A. Chọn cột: giữ lại ['rating', 'title', 'user_id', 'text', 'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
B. Drop NA  : 500,000 → 500,000 (-0)
C. Dtype    : 500,000 → 500,000
D. Timestamp: range 2004 - 2023
E. Sentiment: {'positive': 404943, 'negative': 50452, 'neutral': 44605}
G. Duplicate: 485,592 → 482,244 (-3,348 duplicates)

✅ Kết quả cuối: 482,244 reviews


Streaming Review: 988552it [00:11, 145627.30it/s]

A. Chọn cột: giữ lại ['rating', 'title', 'user_id', 'text', 'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
B. Drop NA  : 500,000 → 500,000 (-0)
C. Dtype    : 500,000 → 500,000
D. Timestamp: range 2004 - 2023
E. Sentiment: {'positive': 395883, 'negative': 57784, 'neutral': 46333}
G. Duplicate: 482,628 → 479,120 (-3,508 duplicates)

✅ Kết quả cuối: 479,120 reviews


...

A. Chọn cột: giữ lại ['rating', 'title', 'user_id', 'text', 'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
B. Drop NA  : 500,000 → 500,000 (-0)
C. Dtype    : 500,000 → 500,000
D. Timestamp: range 2000 - 2023
E. Sentiment: {'positive': 358778, 'negative': 102329, 'neutral': 38893}
G. Duplicate: 472,041 → 440,371 (-31,670 duplicates)

✅ Kết quả cuối: 440,371 reviews


Streaming Review: 65999731it [20:43, 97797.64it/s] 

A. Chọn cột: giữ lại ['rating', 'title', 'user_id', 'text', 'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
B. Drop NA  : 500,000 → 500,000 (-0)
C. Dtype    : 500,000 → 500,000
D. Timestamp: range 2000 - 2023
E. Sentiment: {'positive': 357537, 'negative': 104034, 'neutral': 38429}
G. Duplicate: 470,853 → 454,524 (-16,329 duplicates)

✅ Kết quả cuối: 454,524 reviews


Streaming Review: 66033346it [20:48, 52898.68it/s]


A. Chọn cột: giữ lại ['rating', 'title', 'user_id', 'text', 'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
B. Drop NA  : 33,346 → 33,346 (-0)
C. Dtype    : 33,346 → 33,346
D. Timestamp: range 2000 - 2023
E. Sentiment: {'positive': 23759, 'negative': 7050, 'neutral': 2537}
G. Duplicate: 31,350 → 30,906 (-444 duplicates)

✅ Kết quả cuối: 30,906 reviews
✅ Đã lưu xong file Review cho Sentiment tại: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\review_clean_all.csv


Đọc tiền xử lý + lưu file meta

In [8]:
def process_meta_with_cleaning_chunks(filepath, chunk_size=500000):
    print(f"📦 Đang tiền xử lý file Meta theo cụm {chunk_size:,}...")
    file_exists = False
    
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        current_chunk = []
        for i, line in enumerate(tqdm(f, desc="Streaming Meta")):
            try:
                current_chunk.append(json.loads(line.strip()))
            except: 
                continue

            if len(current_chunk) == chunk_size:
                # 1. Chuyển thành DataFrame
                df_raw = pd.DataFrame(current_chunk)
                
                # 2. Gọi hàm tiền xử lý Meta (Hàm preprocess_meta trong Cell 2.5 của bạn)
                # Hàm này sẽ xử lý: Drop NA, chuẩn hóa giá, chuyển đổi Categories...
                df_temp = preprocess_meta(df_raw)
                
                # 3. Lưu nối đuôi vào file CSV
                df_temp.to_csv(FINAL_META_CSV, mode='a', index=False, header=not file_exists)
                
                file_exists = True
                current_chunk = [] # Giải phóng danh sách tạm
                del df_raw, df_temp # Giải phóng RAM

        # Xử lý nốt phần dữ liệu dư cuối cùng
        if current_chunk:
            df_raw = pd.DataFrame(current_chunk)
            df_temp = preprocess_meta(df_raw)
            df_temp.to_csv(FINAL_META_CSV, mode='a', index=False, header=not file_exists)

    print(f"✅ Đã lưu xong file Meta đã làm sạch tại: {FINAL_META_CSV}")

# Thực thi xử lý file Meta
process_meta_with_cleaning_chunks(META_PATH, CHUNK_SIZE)

📦 Đang tiền xử lý file Meta theo cụm 500,000...


Streaming Meta: 498930it [00:22, 25854.52it/s]

Raw shape: (500000, 16)
A. Giữ cột: ['parent_asin', 'title', 'description', 'categories', 'average_rating', 'rating_number', 'store', 'main_category', 'images', 'price']
B. Drop NA/dup: 500,000 → 500,000
D. Description: đã chuyển list → string
E. Categories top 5: {'Clothing, Shoes & Jewelry': 223172, 'Shoe, Jewelry & Watch Accessories': 59}

✅ Meta kết quả: (223231, 9)


Streaming Meta: 998033it [00:58, 28012.94it/s]

Raw shape: (500000, 16)
A. Giữ cột: ['parent_asin', 'title', 'description', 'categories', 'average_rating', 'rating_number', 'store', 'main_category', 'images', 'price']
B. Drop NA/dup: 500,000 → 500,000
D. Description: đã chuyển list → string
E. Categories top 5: {'Clothing, Shoes & Jewelry': 169221, 'Shoe, Jewelry & Watch Accessories': 70}

✅ Meta kết quả: (169291, 9)


...

Raw shape: (500000, 16)
A. Giữ cột: ['parent_asin', 'title', 'description', 'categories', 'average_rating', 'rating_number', 'store', 'main_category', 'images', 'price']
B. Drop NA/dup: 500,000 → 500,000
D. Description: đã chuyển list → string
E. Categories top 5: {'Clothing, Shoes & Jewelry': 76031, 'Shoe, Jewelry & Watch Accessories': 18}

✅ Meta kết quả: (76049, 9)


Streaming Meta: 6998589it [06:31, 36385.66it/s]

Raw shape: (500000, 16)
A. Giữ cột: ['parent_asin', 'title', 'description', 'categories', 'average_rating', 'rating_number', 'store', 'main_category', 'images', 'price']
B. Drop NA/dup: 500,000 → 500,000
D. Description: đã chuyển list → string
E. Categories top 5: {'Clothing, Shoes & Jewelry': 67183, 'Shoe, Jewelry & Watch Accessories': 14}

✅ Meta kết quả: (67197, 9)


Streaming Meta: 7218481it [06:46, 17746.81it/s]


Raw shape: (218481, 16)
A. Giữ cột: ['parent_asin', 'title', 'description', 'categories', 'average_rating', 'rating_number', 'store', 'main_category', 'images', 'price']
B. Drop NA/dup: 218,481 → 218,481
D. Description: đã chuyển list → string
E. Categories top 5: {'Clothing, Shoes & Jewelry': 27112, 'Shoe, Jewelry & Watch Accessories': 6}

✅ Meta kết quả: (27118, 9)
✅ Đã lưu xong file Meta đã làm sạch tại: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\meta_clean_all.csv


Tạo data base của meta data

In [9]:
DB_PATH = "metadata.db"
# Xóa db cũ nếu có để làm mới
if os.path.exists(DB_PATH): os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)

print("🗄️ Bước 1: Đang chuyển file Meta 12GB vào SQLite...")
# Đọc Meta theo cụm để không tốn RAM
meta_reader = pd.read_csv(FINAL_META_CSV, chunksize=200000, low_memory=False)

for chunk in tqdm(meta_reader, desc="Importing Meta to DB"):
    # Đưa vào bảng 'products'
    chunk.to_sql('products', conn, if_exists='append', index=False)

# TẠO INDEX: Đây là bước quan trọng nhất để tra cứu siêu tốc (O(1))
print("⚡ Đang tạo Index cho parent_asin (để merge nhanh)...")
conn.execute("CREATE INDEX idx_asin ON products (parent_asin)")
conn.close()
print("✅ Đã tạo xong Database Meta trên ổ cứng!")

🗄️ Bước 1: Đang chuyển file Meta 12GB vào SQLite...


Importing Meta to DB: 8it [00:51,  6.44s/it]


⚡ Đang tạo Index cho parent_asin (để merge nhanh)...
✅ Đã tạo xong Database Meta trên ổ cứng!


merge 2 file bằng hybrid (data base + dictionary)

In [15]:
import pyarrow as pa
import pyarrow.parquet as pq
 
MERGED_OUTPUT_PQ = PROCESSED_DIR / "amazon_full_hybrid_merged.parquet"
 
meta_cols_sql = """
    p.title, 
    p.description, 
    p.categories, 
    p.average_rating, 
    p.rating_number, 
    p.store, 
    p.main_category, 
    p.images
"""
 
conn = sqlite3.connect(DB_PATH)
print("🚀 Bắt đầu quá trình Hybrid Merge (SQLite + Pandas)...")
 
review_reader = pd.read_csv(FINAL_REVIEW_CSV, chunksize=CHUNK_SIZE)
pq_writer = None  # Sẽ khởi tạo ở batch đầu tiên
 
for i, chunk in enumerate(tqdm(review_reader, desc="Processing Batches")):
    chunk.to_sql('current_review_batch', conn, if_exists='replace', index=False)
 
    query = f"""
    SELECT r.*, {meta_cols_sql}
    FROM current_review_batch r
    LEFT JOIN products p ON r.parent_asin = p.parent_asin
    """
 
    merged_batch = pd.read_sql_query(query, conn)
    merged_batch = merged_batch.loc[:, ~merged_batch.columns.duplicated()]
 
    # Chuyển DataFrame → PyArrow Table để ghi
    table = pa.Table.from_pandas(merged_batch, preserve_index=False)
 
    # Lần đầu: tạo writer với schema từ batch đầu tiên
    if pq_writer is None:
        pq_writer = pq.ParquetWriter(MERGED_OUTPUT_PQ, table.schema)
 
    pq_writer.write_table(table)
 
    del chunk, merged_batch, table
    gc.collect()
 
# Đóng writer sau khi xong toàn bộ
if pq_writer:
    pq_writer.close()
 
conn.close()
print(f"✨ HOÀN THÀNH! File đã lưu tại: {MERGED_OUTPUT_PQ}")

🚀 Bắt đầu quá trình Hybrid Merge (SQLite + Pandas)...


Processing Batches: 125it [1:23:59, 40.32s/it]

✨ HOÀN THÀNH! File đã lưu tại: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\amazon_full_hybrid_merged.parquet


<span style = "font-size : 30px" >Cold start + lưu <span>

Đơn luồng

In [17]:
INPUT_PARQUET    = PROCESSED_DIR / "amazon_full_hybrid_merged.parquet"
OUTPUT_FINAL_PQ  = PROCESSED_DIR / "amazon_clothing_final_gold.parquet"
K_CORE = 5
 
print(f"🔍 Bắt đầu lọc Cold-start (k={K_CORE}) trên file Parquet...")
 
# --- BƯỚC 1: TÍNH TOÁN ID HỢP LỆ (K-Core) ---
df_ids = pd.read_parquet(INPUT_PARQUET, columns=['user_id', 'parent_asin'])
 
iteration = 0
while True:
    iteration += 1
    n_before = len(df_ids)
 
    item_counts = df_ids['parent_asin'].value_counts()
    valid_items = item_counts[item_counts >= K_CORE].index
    df_ids = df_ids[df_ids['parent_asin'].isin(valid_items)]
 
    user_counts = df_ids['user_id'].value_counts()
    valid_users = user_counts[user_counts >= K_CORE].index
    df_ids = df_ids[df_ids['user_id'].isin(valid_users)]
 
    n_after = len(df_ids)
    print(f"🔄 Vòng lặp {iteration}: {n_before:,} -> {n_after:,} dòng")
 
    if n_before == n_after:
        print("✅ Đã đạt trạng thái ổn định (K-Core convergence).")
        break
 
# --- BƯỚC 2: TẠO SET TRA CỨU NHANH ---
valid_user_set = set(df_ids['user_id'].unique())
valid_item_set = set(df_ids['parent_asin'].unique())
del df_ids
gc.collect()
 
# --- BƯỚC 3: GHI FILE VÀNG BẰNG PYARROW (không dùng fastparquet) ---
print("💾 Đang ghi file 'Gold Dataset' theo từng Row Group...")
parquet_file = pq.ParquetFile(INPUT_PARQUET)
pq_writer = None  # Khởi tạo ở batch đầu tiên
 
for i in tqdm(range(parquet_file.num_row_groups), desc="Streaming Final Filter"):
    chunk = parquet_file.read_row_group(i).to_pandas()
 
    chunk_filtered = chunk[
        chunk['user_id'].isin(valid_user_set) &
        chunk['parent_asin'].isin(valid_item_set)
    ]
 
    if not chunk_filtered.empty:
        table = pa.Table.from_pandas(chunk_filtered, preserve_index=False)
 
        # Lần đầu: tạo writer với schema chuẩn
        if pq_writer is None:
            pq_writer = pq.ParquetWriter(OUTPUT_FINAL_PQ, table.schema)
 
        pq_writer.write_table(table)
 
    del chunk, chunk_filtered
    gc.collect()
 
if pq_writer:
    pq_writer.close()
 
print(f"\n✨ HOÀN THÀNH! File 'Gold Dataset' đã sẵn sàng: {OUTPUT_FINAL_PQ}")

🔍 Bắt đầu lọc Cold-start (k=5) trên file Parquet...
🔄 Vòng lặp 1: 62,381,693 -> 24,052,579 dòng
🔄 Vòng lặp 2: 24,052,579 -> 21,693,321 dòng
🔄 Vòng lặp 3: 21,693,321 -> 21,529,341 dòng
🔄 Vòng lặp 4: 21,529,341 -> 21,516,339 dòng
🔄 Vòng lặp 5: 21,516,339 -> 21,515,311 dòng
🔄 Vòng lặp 6: 21,515,311 -> 21,515,263 dòng
🔄 Vòng lặp 7: 21,515,263 -> 21,515,259 dòng
🔄 Vòng lặp 8: 21,515,259 -> 21,515,259 dòng
✅ Đã đạt trạng thái ổn định (K-Core convergence).
💾 Đang ghi file 'Gold Dataset' theo từng Row Group...


Streaming Final Filter: 100%|██████████| 125/125 [2:09:28<00:00, 62.15s/it] 


✨ HOÀN THÀNH! File 'Gold Dataset' đã sẵn sàng: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\amazon_clothing_final_gold.parquet


Đa luồng

In [4]:
import pyarrow as pa
import pyarrow.parquet as pq
import gc
from concurrent.futures import ThreadPoolExecutor

INPUT_PARQUET   = PROCESSED_DIR / "amazon_full_hybrid_merged.parquet"
OUTPUT_FINAL_PQ = PROCESSED_DIR / "amazon_clothing_final_gold_daluong.parquet"
K_CORE    = 5
N_WORKERS = 4

# =============================================================================
# BƯỚC 1: TÍNH K-CORE (tuần tự)
# =============================================================================
print(f"🔍 Bắt đầu lọc Cold-start (k={K_CORE})...")
df_ids = pd.read_parquet(INPUT_PARQUET, columns=['user_id', 'parent_asin'])

iteration = 0
while True:
    iteration += 1
    n_before = len(df_ids)

    item_counts = df_ids['parent_asin'].value_counts()
    valid_items = item_counts[item_counts >= K_CORE].index
    df_ids = df_ids[df_ids['parent_asin'].isin(valid_items)]

    user_counts = df_ids['user_id'].value_counts()
    valid_users = user_counts[user_counts >= K_CORE].index
    df_ids = df_ids[df_ids['user_id'].isin(valid_users)]

    n_after = len(df_ids)
    print(f"🔄 Vòng lặp {iteration}: {n_before:,} -> {n_after:,} dòng")

    if n_before == n_after:
        print("✅ K-Core hội tụ.")
        break

valid_user_set = set(df_ids['user_id'].unique())
valid_item_set = set(df_ids['parent_asin'].unique())
del df_ids
gc.collect()

# =============================================================================
# BƯỚC 2: ĐỌC + LỌC SONG SONG, GHI TUẦN TỰ
# =============================================================================
parquet_file = pq.ParquetFile(INPUT_PARQUET)
num_rg       = parquet_file.num_row_groups
print(f"\n💾 Xử lý {num_rg} row groups với {N_WORKERS} luồng...")

def process_row_group(rg_index):
    """Đọc + lọc 1 row group, giải phóng RAM ngay trong luồng con."""
    chunk = parquet_file.read_row_group(rg_index).to_pandas()
    filtered = chunk[
        chunk['user_id'].isin(valid_user_set) &
        chunk['parent_asin'].isin(valid_item_set)
    ]
    del chunk  # Giải phóng chunk gốc ngay

    if filtered.empty:
        del filtered
        gc.collect()  # Dọn ngay trong luồng con, không chờ GIL
        return None

    table = pa.Table.from_pandas(filtered, preserve_index=False)
    del filtered
    gc.collect()  # Dọn ngay sau khi đã convert sang Arrow
    return table

pq_writer = None

with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    # executor.map: xử lý song song, trả kết quả đúng thứ tự
    for table in tqdm(
        executor.map(process_row_group, range(num_rg)),
        total=num_rg,
        desc="Processing"
    ):
        if table is not None:
            if pq_writer is None:
                pq_writer = pq.ParquetWriter(OUTPUT_FINAL_PQ, table.schema)
            pq_writer.write_table(table)
            del table
            gc.collect()  # Dọn sau khi ghi xong

if pq_writer:
    pq_writer.close()

print(f"\n✨ HOÀN THÀNH! Gold Dataset: {OUTPUT_FINAL_PQ}")

🔍 Bắt đầu lọc Cold-start (k=5)...
🔄 Vòng lặp 1: 62,381,693 -> 24,052,579 dòng
🔄 Vòng lặp 2: 24,052,579 -> 21,693,321 dòng
🔄 Vòng lặp 3: 21,693,321 -> 21,529,341 dòng
🔄 Vòng lặp 4: 21,529,341 -> 21,516,339 dòng
🔄 Vòng lặp 5: 21,516,339 -> 21,515,311 dòng
🔄 Vòng lặp 6: 21,515,311 -> 21,515,263 dòng
🔄 Vòng lặp 7: 21,515,263 -> 21,515,259 dòng
🔄 Vòng lặp 8: 21,515,259 -> 21,515,259 dòng
✅ K-Core hội tụ.

💾 Xử lý 125 row groups với 4 luồng...


Processing: 100%|██████████| 125/125 [17:59<00:00,  8.63s/it]


✨ HOÀN THÀNH! Gold Dataset: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\amazon_clothing_final_gold_daluong.parquet
